In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\results_clean.csv")
elo = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\elo_ratings.csv")
conf = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\confederation_map.csv")
ranks = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\fifa_rankings.csv")

print("results:", df.shape)
print("elo:", elo.shape)
print("confederations:", conf.shape)
print("rankings:", ranks.shape)


results: (32101, 11)
elo: (48, 2)
confederations: (79, 2)
rankings: (48, 4)


In [4]:
# Load full elo file
elo_all = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\elo_all_teams.csv")

# Merge home team Elo
df = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\results_clean.csv")
df = df.merge(elo_all[['team', 'elo']], left_on='home_team', right_on='team', how='left')
df = df.rename(columns={'elo': 'home_elo'}).drop(columns='team')

# Merge away team Elo
df = df.merge(elo_all[['team', 'elo']], left_on='away_team', right_on='team', how='left')
df = df.rename(columns={'elo': 'away_elo'}).drop(columns='team')

print(df[['home_team', 'home_elo', 'away_team', 'away_elo']].head())
print(f"\nMissing home_elo: {df['home_elo'].isnull().sum()}")
print(f"Missing away_elo: {df['away_elo'].isnull().sum()}")

  home_team     home_elo  away_team     away_elo
0   Algeria  1837.945150       Mali  1646.944724
1   Algeria  1837.945150   Cameroon  1731.465033
2    Greece  1726.773470    Belgium  1780.811104
3    Mexico  1911.874693  Argentina  2047.584193
4    Malawi  1384.462883   Tanzania  1388.469275

Missing home_elo: 0
Missing away_elo: 0


In [5]:
df['elo_diff'] = df['home_elo'] - df['away_elo']

print(df[['home_team', 'home_elo', 'away_team', 'away_elo', 'elo_diff']].head())

  home_team     home_elo  away_team     away_elo    elo_diff
0   Algeria  1837.945150       Mali  1646.944724  191.000426
1   Algeria  1837.945150   Cameroon  1731.465033  106.480117
2    Greece  1726.773470    Belgium  1780.811104  -54.037634
3    Mexico  1911.874693  Argentina  2047.584193 -135.709501
4    Malawi  1384.462883   Tanzania  1388.469275   -4.006393


In [6]:
# Merge home confederation
df = df.merge(conf, left_on='home_team', right_on='team', how='left')
df = df.rename(columns={'confederation': 'home_conf'}).drop(columns='team')

# Merge away confederation
df = df.merge(conf, left_on='away_team', right_on='team', how='left')
df = df.rename(columns={'confederation': 'away_conf'}).drop(columns='team')

print(df[['home_team', 'home_conf', 'away_team', 'away_conf']].head())
print(f"\nMissing home_conf: {df['home_conf'].isnull().sum()}")
print(f"Missing away_conf: {df['away_conf'].isnull().sum()}")

  home_team home_conf  away_team away_conf
0   Algeria       CAF       Mali       CAF
1   Algeria       CAF   Cameroon       CAF
2    Greece       NaN    Belgium      UEFA
3    Mexico  CONCACAF  Argentina  CONMEBOL
4    Malawi       NaN   Tanzania       CAF

Missing home_conf: 15957
Missing away_conf: 16760


In [7]:
# Extended confederation map for common missing teams
extra_conf = {
    # UEFA
    'Greece': 'UEFA', 'Russia': 'UEFA', 'Sweden': 'UEFA', 'Norway': 'UEFA',
    'Finland': 'UEFA', 'Ireland': 'UEFA', 'Wales': 'UEFA', 'Northern Ireland': 'UEFA',
    'Iceland': 'UEFA', 'Bulgaria': 'UEFA', 'Montenegro': 'UEFA', 'Kosovo': 'UEFA',
    'North Macedonia': 'UEFA', 'Moldova': 'UEFA', 'Belarus': 'UEFA',
    'Azerbaijan': 'UEFA', 'Armenia': 'UEFA', 'Kazakhstan': 'UEFA',
    'Lithuania': 'UEFA', 'Latvia': 'UEFA', 'Estonia': 'UEFA',
    'Luxembourg': 'UEFA', 'Malta': 'UEFA', 'Cyprus': 'UEFA',
    'Liechtenstein': 'UEFA', 'Andorra': 'UEFA', 'San Marino': 'UEFA',
    'Faroe Islands': 'UEFA', 'Gibraltar': 'UEFA',
    # CAF
    'Zambia': 'CAF', 'Zimbabwe': 'CAF', 'Kenya': 'CAF', 'Uganda': 'CAF',
    'Ethiopia': 'CAF', 'Rwanda': 'CAF', 'Mozambique': 'CAF', 'Malawi': 'CAF',
    'Burkina Faso': 'CAF', 'Guinea': 'CAF', 'Guinea-Bissau': 'CAF',
    'Equatorial Guinea': 'CAF', 'Gabon': 'CAF', 'Congo': 'CAF',
    'Togo': 'CAF', 'Niger': 'CAF', 'Chad': 'CAF', 'Sudan': 'CAF',
    'Libya': 'CAF', 'Mauritania': 'CAF', 'Gambia': 'CAF',
    'Sierra Leone': 'CAF', 'Liberia': 'CAF', 'Namibia': 'CAF',
    'Botswana': 'CAF', 'Lesotho': 'CAF', 'Eswatini': 'CAF',
    'Madagascar': 'CAF', 'Mauritius': 'CAF', 'Djibouti': 'CAF',
    'Somalia': 'CAF', 'Eritrea': 'CAF', 'South Sudan': 'CAF',
    'Central African Republic': 'CAF', 'Cameroon': 'CAF',
    # AFC
    'China': 'AFC', 'India': 'AFC', 'Thailand': 'AFC', 'Vietnam': 'AFC',
    'Malaysia': 'AFC', 'Philippines': 'AFC', 'Myanmar': 'AFC',
    'Singapore': 'AFC', 'Hong Kong': 'AFC', 'Taiwan': 'AFC',
    'North Korea': 'AFC', 'Kuwait': 'AFC', 'UAE': 'AFC',
    'Bahrain': 'AFC', 'Lebanon': 'AFC', 'Syria': 'AFC', 'Yemen': 'AFC',
    'Palestine': 'AFC', 'Kyrgyzstan': 'AFC', 'Tajikistan': 'AFC',
    'Turkmenistan': 'AFC', 'Afghanistan': 'AFC', 'Nepal': 'AFC',
    'Bangladesh': 'AFC', 'Sri Lanka': 'AFC', 'Pakistan': 'AFC',
    'Maldives': 'AFC', 'Bhutan': 'AFC', 'Mongolia': 'AFC',
    'Macau': 'AFC', 'Cambodia': 'AFC', 'Laos': 'AFC', 'Timor-Leste': 'AFC',
    # CONCACAF
    'Barbados': 'CONCACAF', 'Belize': 'CONCACAF', 'Bermuda': 'CONCACAF',
    'Nicaragua': 'CONCACAF', 'Dominican Republic': 'CONCACAF',
    'Puerto Rico': 'CONCACAF', 'Antigua and Barbuda': 'CONCACAF',
    'Saint Kitts and Nevis': 'CONCACAF', 'Saint Lucia': 'CONCACAF',
    'Saint Vincent and the Grenadines': 'CONCACAF', 'Grenada': 'CONCACAF',
    'Dominica': 'CONCACAF', 'Guyana': 'CONCACAF', 'Suriname': 'CONCACAF',
    'Aruba': 'CONCACAF', 'Cayman Islands': 'CONCACAF',
    # CONMEBOL
    'Peru': 'CONMEBOL',
    # OFC
    'Fiji': 'OFC', 'Papua New Guinea': 'OFC', 'Solomon Islands': 'OFC',
    'Vanuatu': 'OFC', 'Tahiti': 'OFC', 'New Caledonia': 'OFC',
    'Samoa': 'OFC', 'American Samoa': 'OFC', 'Tonga': 'OFC',
    'Cook Islands': 'OFC',
}

# Add to existing conf dataframe and re-merge
extra_df = pd.DataFrame(list(extra_conf.items()), columns=['team', 'confederation'])
conf_full = pd.concat([conf, extra_df], ignore_index=True).drop_duplicates(subset='team')

# Re-merge
df = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\results_clean.csv")
df = df.merge(elo_all[['team', 'elo']], left_on='home_team', right_on='team', how='left')
df = df.rename(columns={'elo': 'home_elo'}).drop(columns='team')
df = df.merge(elo_all[['team', 'elo']], left_on='away_team', right_on='team', how='left')
df = df.rename(columns={'elo': 'away_elo'}).drop(columns='team')
df['elo_diff'] = df['home_elo'] - df['away_elo']

df = df.merge(conf_full, left_on='home_team', right_on='team', how='left')
df = df.rename(columns={'confederation': 'home_conf'}).drop(columns='team')
df = df.merge(conf_full, left_on='away_team', right_on='team', how='left')
df = df.rename(columns={'confederation': 'away_conf'}).drop(columns='team')

print(f"Missing home_conf: {df['home_conf'].isnull().sum()}")
print(f"Missing away_conf: {df['away_conf'].isnull().sum()}")
df[['home_team', 'home_conf', 'away_team', 'away_conf']].head()

Missing home_conf: 2735
Missing away_conf: 2710


,home_team,home_conf,away_team,away_conf
0,Algeria,CAF,Mali,CAF
1,Algeria,CAF,Cameroon,CAF
2,Greece,UEFA,Belgium,UEFA
3,Mexico,CONCACAF,Argentina,CONMEBOL
4,Malawi,CAF,Tanzania,CAF


In [8]:
df['home_conf'] = df['home_conf'].fillna('OTHER')
df['away_conf'] = df['away_conf'].fillna('OTHER')

print(f"Missing home_conf: {df['home_conf'].isnull().sum()}")
print(f"Missing away_conf: {df['away_conf'].isnull().sum()}")
print(f"\nConfederation counts:\n{df['home_conf'].value_counts()}")

Missing home_conf: 0
Missing away_conf: 0

Confederation counts:
home_conf
UEFA        9036
CAF         7126
AFC         6490
CONCACAF    4155
OTHER       2735
CONMEBOL    1925
OFC          634
Name: count, dtype: int64


In [9]:
# Merge home FIFA rank
df = df.merge(ranks[['Nation', 'FIFA_2026_rank']], left_on='home_team', right_on='Nation', how='left')
df = df.rename(columns={'FIFA_2026_rank': 'home_fifa_rank'}).drop(columns='Nation')

# Merge away FIFA rank
df = df.merge(ranks[['Nation', 'FIFA_2026_rank']], left_on='away_team', right_on='Nation', how='left')
df = df.rename(columns={'FIFA_2026_rank': 'away_fifa_rank'}).drop(columns='Nation')

# Fill missing ranks with a neutral value (25 = middle of 48 teams)
df['home_fifa_rank'] = df['home_fifa_rank'].fillna(25)
df['away_fifa_rank'] = df['away_fifa_rank'].fillna(25)

# Rank difference — negative means home team is ranked higher (better)
df['fifa_rank_diff'] = df['home_fifa_rank'] - df['away_fifa_rank']

print(df[['home_team', 'home_fifa_rank', 'away_team', 'away_fifa_rank', 'fifa_rank_diff']].head())

  home_team  home_fifa_rank  away_team  away_fifa_rank  fifa_rank_diff
0   Algeria            28.0       Mali            25.0             3.0
1   Algeria            28.0   Cameroon            25.0             3.0
2    Greece            25.0    Belgium             9.0            16.0
3    Mexico            15.0  Argentina             3.0            12.0
4    Malawi            25.0   Tanzania            25.0             0.0


In [10]:
print(df.shape)
print(df.columns.tolist())
df.head(3)

(32101, 19)
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'tournament_weight', 'result', 'home_elo', 'away_elo', 'elo_diff', 'home_conf', 'away_conf', 'home_fifa_rank', 'away_fifa_rank', 'fifa_rank_diff']


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,result,home_elo,away_elo,elo_diff,home_conf,away_conf,home_fifa_rank,away_fifa_rank,fifa_rank_diff
0,1990-01-12,Algeria,Mali,5,0,Friendly,Paris,France,True,0.5,H,1837.94515,1646.944724,191.000426,CAF,CAF,28.0,25.0,3.0
1,1990-01-14,Algeria,Cameroon,3,1,Friendly,Paris,France,True,0.5,H,1837.94515,1731.465033,106.480117,CAF,CAF,28.0,25.0,3.0
2,1990-01-17,Greece,Belgium,2,0,Friendly,Athens,Greece,False,0.5,H,1726.77347,1780.811104,-54.037634,UEFA,UEFA,25.0,9.0,16.0


In [11]:
df.to_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\features.csv", index=False)
print("Saved!")

Saved!
